In [ ]:
#OBSOLETE test_vis_params_manual
#NOTE: this is a hodgepodge - not cleaned up and/or working - see test_vis_params
import ee
import geemap
import pandas as pd
import os
from shapely.geometry import box, Polygon
import matplotlib.pyplot as plt
import numpy as np

#this or ee.Initialize
Map = geemap.Map()

folder_base = r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics' #os.path.join()
folder_shp = r'C:\Users\andyb\Documents\U\GEE-Courses\data'
file_path=os.path.join(folder_base,'glacierPropsLandsat.csv')

glaciers = pd.read_csv(file_path) #contains Name, LatCenter, LonCenter, two types of bounding boxes (see GD_Landsat_01_Setup).
glaciers['Name']

#choose one glacier 
glacier = glaciers.iloc[0]
glacierdf=glaciers.iloc[[0]]
print('You chose: ' + glacier['Name'])
folder_out=os.path.join(folder_shp, glacier['Name'])
folder_fig=os.path.join(folder_out, 'Figures')
os.makedirs(folder_out, exist_ok=True)
os.makedirs(folder_fig, exist_ok=True)

glacierPt = ee.Geometry.Point(glacier['LonCenter'],glacier['LatCenter']) #-137.121, 58.838) #Johns Hopkins (center of terminus)
#TODO: explore polygon instead (may work seamlessly, may not)
#SEE ALSO: reducing/clipping to area in "create an image composite" section of reducing_image_collection.ipynb
Map.centerObject(glacierPt, 12)  # Zoom level 12 for a close view
# Add a marker at the point (optional, for visualization)
Map.addLayer(glacierPt, {'color': 'red'}, glacier['Name'])

#add region of interest
#roi = ee.Geometry.Rectangle([glacier['LonMin'], glacier['LatMin'], glacier['LonMax'], glacier['LatMax']])
def load_polygon(df):
    coords = []
    i = 1
    while f'x{i}' in df.columns and f'y{i}' in df.columns:
        x = df[f'x{i}'].iloc[0]
        y = df[f'y{i}'].iloc[0]
        coords.append((x, y))
        i += 1
    # Ensure the polygon is closed (first and last points are the same)
    if coords and coords[0] != coords[-1]:
        coords.append(coords[0])
    return Polygon(coords)

roi=load_polygon(glacierdf)
#shapely back to ee_polygon: Get the exterior coordinates as a list of lists
roicoords = [list(roi.exterior.coords)]
roi_ee = ee.Geometry.Polygon(roicoords)
print(roi)
#print(roi_ee)

collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterDate('2014-01-01', '2015-01-01')
    .filterBounds(glacierPt)
    .sort('system:time_start')
)


# Add the Landsat TOA image to the map (visualize with true color bands) TOP OF ATMOSPHERE
vis_params = {
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue for TOA
    'min': 0.05,
    'max': 1.6 #was 0.3 or 0. for normal land scenes, not snow
}
#GEEDiT uses bands:['B3','B2','B1'],gamma:1.5,min:0,max:0.8
vis_params45 = {
    'bands': ['B3', 'B2', 'B1'],  # Red, Green, Blue for TOA
    'min': 0.05,
    'max': 1.6
}

vis_true = { #example in test_landsat_geemap used SR, here we use TOA, so delete the SR_ prefix.
    'bands': ['B4', 'B3', 'B2'],   # Red, Green, Blue
    'min': 0.0,
    'max': 0.3,
    'gamma': 1.4
}

vis_false = {
    'bands': ['B5', 'B4', 'B3'],   # NIR, Red, Green
    'min': 0.0,
    'max': 0.3,
    'gamma': 1.4
}
Map.addLayer(collection_scaled.first(), vis_trueScaled, "ScaleFirst "+mdfs['DATE_ACQUIRED'][0])
Map.addLayer(collection_scaled.first(), vis_falseScaled, "ScaleFirst false "+mdfs['DATE_ACQUIRED'][0])
Map.addLayer(collection.filterDate('2021-12-01', '2021-12-5'),vis_params, "vis_params ~2021-12-1") #just added, untested.
Map.addLayer(collection.first(), vis_params, "First params"+mdf['DATE_ACQUIRED'][0])
